In [ ]:
import os, glob
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from pycaret.clustering import setup, create_model, assign_model
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from datetime import datetime
# ---------------------------------------------------------
# 1. 데이터 로드 (경주님이 만든 함수 사용)
# ---------------------------------------------------------
def load_file_data(path):
    data_list = []
    if not path:
        raise ValueError("path가 비어 있습니다. 파일 경로 리스트를 전달해야 합니다.")
    if not isinstance(path, list):
        raise TypeError("path는 list 타입이어야 합니다.")

    for file_ in path:
        filename = os.path.basename(file_).split('.')[0]
        try:
            with open(file_, 'r', encoding='latin1') as f:
                text_content = f.read()
            data_list.append({'filename': filename, 'opinion_text': text_content})
        except FileNotFoundError:
            print(f"Error: {file_} not found.")
        except Exception as e:
            print(f"An error occurred reading {file_}: {e}")

    return pd.DataFrame(data_list)


# 실제 데이터 로드
path = r'../data'
all_files = glob.glob(os.path.join(path, "*.data"))   # 확장자 확인 필요
document_df = load_file_data(all_files)


# ---------------------------------------------------------
# 2. TF-IDF 벡터화
# ---------------------------------------------------------
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X = vectorizer.fit_transform(document_df['opinion_text'].astype(str))
tfidf_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())

# 저장

# ---------------------------------------------------------
# 3. PyCaret Clustering 환경 설정 및 모델 생성
# ---------------------------------------------------------
s = setup(data=tfidf_df, session_id=42)

kmeans = create_model('kmeans')
dbscan = create_model('dbscan')
birch = create_model('birch')


# ---------------------------------------------------------
# 4. 실루엣 점수 계산
# ---------------------------------------------------------
def get_silhouette_score(model, tfidf_df):
    clustered = assign_model(model)
    labels = clustered['Cluster']
    if len(set(labels)) > 1 and -1 not in set(labels):
        return silhouette_score(tfidf_df, labels)
    else:
        return None

kmeans_score = get_silhouette_score(kmeans, tfidf_df)
dbscan_score = get_silhouette_score(dbscan, tfidf_df)
birch_score = get_silhouette_score(birch, tfidf_df)

scores = {
    "KMeans": kmeans_score if kmeans_score is not None else 0,
    "DBSCAN": dbscan_score if dbscan_score is not None else 0,
    "Birch": birch_score if birch_score is not None else 0
}


# ---------------------------------------------------------
# 5. 결과 저장 (텍스트 + 그래프)
# ---------------------------------------------------------
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
os.makedirs("../results", exist_ok=True)
os.makedirs("../images", exist_ok=True)


# 텍스트 저장
results_path = f"../results/results_{timestamp}.txt"
with open(results_path, "w", encoding="utf-8") as f:
    f.write("=== Silhouette Score 비교 ===\n")
    for model, score in scores.items():
        f.write(f"{model}: {score}\n")

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import pandas as pd


# --- Silhouette Score 막대 그래프 ---
print(scores)  # 값 확인
plt.figure(figsize=(6,4))
plt.bar(scores.keys(), scores.values(), color=['skyblue','salmon','lightgreen'])
plt.ylabel("Silhouette Score")
plt.title("Clustering Model Comparison")
plt.show()

# --- PCA 시각화 (KMeans, DBSCAN, Birch 각각) ---
pca = PCA(n_components=2)
X_pca = pca.fit_transform(tfidf_df)

for model_name, model in zip(["KMeans","DBSCAN","Birch"], [kmeans, dbscan, birch]):
    clustered = assign_model(model)
    labels = pd.to_numeric(clustered['Cluster'], errors='coerce')
    
    plt.figure(figsize=(6,5))
    plt.scatter(X_pca[:,0], X_pca[:,1], c=labels, cmap='tab10')
    plt.title(f"PCA Visualization ({model_name} Clusters)")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.show()

# --- t-SNE 시각화 (KMeans, DBSCAN, Birch 각각) ---
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(tfidf_df)

for model_name, model in zip(["KMeans","DBSCAN","Birch"], [kmeans, dbscan, birch]):
    clustered = assign_model(model)
    labels = pd.to_numeric(clustered['Cluster'], errors='coerce')
    
    plt.figure(figsize=(6,5))
    plt.scatter(X_tsne[:,0], X_tsne[:,1], c=labels, cmap='tab10')
    plt.title(f"t-SNE Visualization ({model_name} Clusters)")
    plt.xlabel("t-SNE1")
    plt.ylabel("t-SNE2")
    plt.show()

